## Document Translation


For Azure Document Translation to work properly, the service needs to be able to read from your source container and write to your target container. Here's how permissions work:

1. **Source Container**: Needs `read` and `list` permissions so the translation service can access the files
2. **Target Container**: Needs `write`, `create`, `read`, and `list` permissions for the service to write translated documents

The most common approaches to grant these permissions are:

- **SAS (Shared Access Signature) tokens**: Temporary, limited-privilege access to your storage
- **Azure AD integration**: Using managed identities (most secure for production)


This notebook demonstrates both the SAS token approach and public access approach.

SDK python code for azure documment translation  and text service are available in this following links:
- https://github.com/Azure/azure-sdk-for-python/tree/azure-ai-translation-text_1.0.1/sdk/translation/azure-ai-translation-document
- https://github.com/Azure/azure-sdk-for-python/tree/azure-ai-translation-text_1.0.1/sdk/translation/azure-ai-translation-text/

# Azure Managed Identity Authentication

Azure Managed Identity allows your application or notebook to authenticate securely without storing sensitive keys. However, the identity itself doesn't automatically have permissions to access Azure resources. You must explicitly grant permissions by assigning Azure RBAC (Role-Based Access Control) roles.

## Azure Services Involved

In your scenario, you have two Azure services involved:

- **Azure Blob Storage**: To read/write documents.
- **Azure Document Translation (Cognitive Services)**: To translate documents.

## Roles Needed

| Azure Service | Required Role | Reason |
|---------------|--------------|---------|
| Blob Storage | Storage Blob Data Contributor | Allows your Managed Identity to read, write, and manage blobs in your storage containers. |
| Document Translation (Cognitive Services) | Cognitive Services User | Allows your Managed Identity to use Azure Cognitive Services APIs (like Document Translation). |

## Step-by-Step Guide to Assign Roles

### Step 1: Identify Your Managed Identity or User Object ID

#### Managed Identity
If you're using a Managed Identity (e.g., from Azure VM, Azure Functions, Azure App Service), find its Object ID in Azure Portal:

1. Go to your Azure resource (e.g., VM, App Service).
2. Click on **Identity**.
3. Copy the **Object ID**.

#### Your Azure User Account
If you're running locally (like your notebook), your Azure CLI login (`az login`) uses your Azure user account. To find your user Object ID, run:

```bash
az ad signed-in-user show --query id --output tsv
```

### Step 2: Assign Roles via Azure CLI

Replace placeholders (`<your-object-id>`, `<subscription-id>`, `<resource-group>`, `<storage-account-name>`, `<translator-resource-name>`) with your actual values.

#### Assign Storage Blob Data Contributor Role:

```bash
az role assignment create \
  --assignee "<your-object-id>" \
  --role "Storage Blob Data Contributor" \
  --scope "/subscriptions/<subscription-id>/resourceGroups/<resource-group>/providers/Microsoft.Storage/storageAccounts/<storage-account-name>"
```

#### Assign Cognitive Services User Role (for Document Translation):

```bash
az role assignment create \
  --assignee "<your-object-id>" \
  --role "Cognitive Services User" \
  --scope "/subscriptions/<subscription-id>/resourceGroups/<resource-group>/providers/Microsoft.CognitiveServices/accounts/<translator-resource-name>"
```

### Step 3: Assign Roles via Azure Portal (Alternative)

1. Navigate to Azure Portal: https://portal.azure.com
2. Go to your Resource Group containing your resources.
3. Click on your Storage Account:
   - Select **Access Control (IAM)** → **Add** → **Add role assignment**.
   - Select role: **Storage Blob Data Contributor**.
   - Under "Members," select your Managed Identity or your Azure AD user.
   - Click **Review + Assign**.

4. Repeat for Cognitive Services (Translator):
   - Go to your Translator resource in Azure Portal.
   - Click **Access Control (IAM)**.
   - Click **Add** → **Add role assignment**.
   - Select **Cognitive Services User** role.
   - Add your Managed Identity or Azure AD user.
   - Click **Review + Assign**.

### Step 4: Verify Permissions

To verify that your identity has the correct permissions:

```bash
# List role assignments for your user
az role assignment list --assignee <your-object-id> --output table

# Or check roles for a specific resource
az role assignment list --scope "/subscriptions/<subscription-id>/resourceGroups/<resource-group>/providers/Microsoft.Storage/storageAccounts/<storage-account-name>" --output table
```

### Import Libraries


In [ ]:
%pip install openai
%pip install python-dotenv
%pip install azure-ai-translation-document==1.0.0
%pip install azure-core
# Install Azure Blob Storage libraries
%pip install azure-storage-blob
# Install Azure Identity for managed identity support
%pip install azure-identity

In [4]:
targetLanguage = 'es'  # French fr -  ga for irish - es for spanish - sv for swedish link fo Supported languages https://learn.microsoft.com/en-us/azure/ai-services/translator/language-support 
local_md_file = '../data/hachette/ScreenTime/Screen Time - Lisa Guernsey_full_text.md'

In [ ]:
# Import required libraries
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient, BlobClient
from azure.ai.translation.document import DocumentTranslationClient
import os
import datetime
from dotenv import load_dotenv

load_dotenv("../keys.env")

# Azure resources configuration
translator_endpoint = os.getenv('AZURE_TRANSLATOR_DOCUMENT_ENDPOINT')
storage_account_name = os.getenv('AZURE_STORAGE_ACCOUNT_NAME')

# Verify required environment variables
if not all([translator_endpoint, storage_account_name]):
    raise ValueError("Missing required environment variables. Check your .env file.")

# Authenticate using Managed Identity (DefaultAzureCredential)
credential = DefaultAzureCredential()

# Blob Storage client using Managed Identity
blob_service_client = BlobServiceClient(
    account_url=f"https://{storage_account_name}.blob.core.windows.net",
    credential=credential
)

# Define container names
source_container_name = "source-documents"
target_container_name = "translated-documents"

for container_name in [source_container_name, target_container_name]:
    container_client = blob_service_client.get_container_client(container_name)
    try:
        container_client.get_container_properties()
        print(f"Container {container_name} already exists")
    except Exception:
        print(f"Creating container: {container_name}")
        blob_service_client.create_container(container_name, public_access=None)

# Upload document to source container
blob_name = os.path.basename(local_md_file)
blob_client = blob_service_client.get_blob_client(container=source_container_name, blob=blob_name)

with open(local_md_file, "rb") as data:
    blob_client.upload_blob(data, overwrite=True)
    print(f"Uploaded {blob_name} to {source_container_name} container")

# Generate unique folder name for translation output
timestamp = datetime.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
target_folder = f"translation-{timestamp}"

source_url = f"https://{storage_account_name}.blob.core.windows.net/{source_container_name}"
target_url = f"https://{storage_account_name}.blob.core.windows.net/{target_container_name}/{targetLanguage}/{timestamp}"

# Document Translation client using Managed Identity
translation_client = DocumentTranslationClient(endpoint=translator_endpoint, credential=credential)

# Start translation
print("Waiting for translation to complete (this may take time)...")
poller = translation_client.begin_translation(
    source_url=source_url,
    target_url=target_url,
    target_language=targetLanguage
)

result = poller.result()

# Display translation details
print(f'Status: {poller.status()}')
print(f'Total documents: {poller.details.documents_total_count}')
print(f'{poller.details.documents_succeeded_count} succeeded, {poller.details.documents_failed_count} failed')

# Download translated documents
for document in result:
    if document.status == 'Succeeded':
        translated_blob_url = document.translated_document_url
        translated_blob_client = BlobClient.from_blob_url(translated_blob_url, credential=credential)

        original_filename_without_ext = os.path.splitext(blob_name)[0]
        local_translated_filename = f"{original_filename_without_ext}_en_to_{targetLanguage}.md"
        local_translated_filepath = os.path.join("../data/hachette/translated", local_translated_filename)

        os.makedirs(os.path.dirname(local_translated_filepath), exist_ok=True)

        with open(local_translated_filepath, "wb") as local_file:
            download_stream = translated_blob_client.download_blob()
            local_file.write(download_stream.readall())

        print(f"Translated document downloaded successfully to {local_translated_filepath}")
    else:
        print(f'Document failed: {document.error.code}, {document.error.message}')